# Benchmark: contiguous $T_l/T_p$ lag consumption

This notebook compares the committed analytic-parity production kernel with an experimental route that stores $T_l$ and $T_p$ in lag-consumption order and reads each lag block through a contiguous slice.

The physical and numerical calculation is unchanged. Both routes use the same full-year WDM grid, six materialized one-arm inputs, $L=25$, complex64 arithmetic, and the tuned `row_chunk_size=32640`, `lag_block_size=17` schedule.

The experiment changes only the memory layout and access pattern of $T_l/T_p$. The production route remains untouched unless the candidate passes the numerical, median-improvement, and IQR-separation guards.


In [ ]:
# --- 1. Imports, environment, and source-file checks ---

import gc
import inspect
import os
import platform
import sys
import time
from datetime import datetime
from itertools import product

# ------------------------------------------------------------------
# Backend selection
#
# IMPORTANT:
# This must be set before importing jax, jax.numpy, FEW, WDM,
# WDM_LISAresponse, or anything else that may import JAX.
# ------------------------------------------------------------------

os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import numpy as np

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import psutil
except ImportError:
    psutil = None

from few.utils.constants import YRSID_SI
from few.waveform import GenerateEMRIWaveform
from lisaorbits.file import read_orbit_file
from lisaorbits import InterpolatedOrbits

from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)
from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    _build_wdm_dT_grid,
    _build_wdm_dT_context_from_wdm,
)
from WDM_LISAresponse.code.lisa_response.one_arm_plan import (
    OneArmWDMPlan,
    _combine_polarizations_device,
    _normalise_one_arm_execution_mode,
)

import WDM_LISAresponse.code.lisa_response.one_arm_plan as one_arm_plan_module
import WDM_LISAresponse.code.lisa_response.response_with_plan as response_with_plan_module
import WDM_LISAresponse
import WDM


_process = psutil.Process(os.getpid()) if psutil is not None else None


def rss_gib():
    if _process is None:
        return np.nan
    return _process.memory_info().rss / 1024**3


print("Timestamp:", datetime.now().isoformat(timespec="seconds"))
print("Python executable:", sys.executable)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("JAX:", jax.__version__)
print("Requested JAX platforms:", os.environ.get("JAX_PLATFORMS"))
print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())
print("WDM:", WDM.__file__)
print("WDM_LISAresponse:", WDM_LISAresponse.__file__)
print("one_arm_plan:", one_arm_plan_module.__file__)
print("response_with_plan:", response_with_plan_module.__file__)
print(
    "OneArmWDMPlan.apply_device:",
    inspect.signature(OneArmWDMPlan.apply_device),
)
print("RSS:", f"{rss_gib():.2f} GiB")

assert jax.default_backend() == "cpu", (
    "JAX did not initialize the CPU backend. Restart the kernel and "
    "run this cell first, before any JAX-related imports."
)

assert all(device.platform == "cpu" for device in jax.devices())

assert (
    _normalise_one_arm_execution_mode("materialized")
    == "materialized"
)
assert (
    "execution_mode"
    in inspect.signature(OneArmWDMPlan.apply_device).parameters
)

assert hasattr(
    baseline_method_owner := __import__(
        "WDM.code.time_delay_filters.plans",
        fromlist=["VariableShiftBatchPlan"],
    ).VariableShiftBatchPlan,
    "apply_device_contiguous_tltp",
), (
    "The imported WDM package does not contain the contiguous-Tl/Tp "
    "experimental plan method. Check that the corrected plans.py and "
    "_time_shift_jax.py files are on the active branch, then restart "
    "the kernel."
)


In [ ]:
# --- 2. Scientific configuration and benchmark controls ---

# Full-year production-like case.
N = 3_200_000
dt = 10
Nf = 100
q_wdm = 4
d_wdm = 4
A_frac = 0.25
L_TRUNC = 25

# Windowing.
alpha = 0.005
tukey_factor = 4

# Orbit start and sky position.
ORBIT_FILE = 'esa-trailing-orbits.h5'
theta = np.pi / 2
phi = 0.0

# EMRI parameters.
m1 = 1e6
m2 = 10.0
a = 0.9
p0 = 8.5
e0 = 0.3
x0 = 1.0
dist = 1.0
qK = np.pi / 3
phiK = np.pi / 3
Phi_phi0 = np.pi / 3
Phi_theta0 = 0.0
Phi_r0 = np.pi / 3
eps = 1e-2

# Fixed production numerical settings.
BASE_SHIFT_KWARGS = {
    "assembly_backend": "lagfirst_chunked_lagblock",
    "assembly_precision": "complex64",
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
    "use_jax": True,
}

ROW_CHUNK_SIZE = 32640
LAG_BLOCK_SIZE = 17

CONFIRM_WARMUPS = 2
CONFIRM_REPEATS = 15
MIN_MEDIAN_IMPROVEMENT_PERCENT = 3.0
MAX_REL_L2 = 1e-6

SAVE_RESULTS = False
RESULTS_PREFIX = "one_arm_contiguous_tltp"

print('Full-year samples:', N)
print('Expected WDM Nt:', N // (2 * Nf))
print('L_trunc:', L_TRUNC, '| number of lags:', 2 * L_TRUNC + 1)
print('Fixed row chunk:', ROW_CHUNK_SIZE)
print('Fixed lag block:', LAG_BLOCK_SIZE)
print(
    'Adoption guard:',
    f'median improvement >= {MIN_MEDIAN_IMPROVEMENT_PERCENT:.1f}%',
    'and candidate q75 < baseline q25',
)


In [ ]:
# --- 3. Timing, validation, and display helpers ---

def block_result(value):
    # Synchronize all JAX leaves before stopping a timer.
    for leaf in jax.tree_util.tree_leaves(value):
        if hasattr(leaf, 'block_until_ready'):
            leaf.block_until_ready()
    return value


def summarize_timings(timings):
    timings = np.asarray(timings, dtype=float)
    if timings.size == 0:
        raise ValueError('At least one timing is required.')

    q25_s, q75_s = np.percentile(timings, [25.0, 75.0])

    return {
        'median_s': float(np.median(timings)),
        'minimum_s': float(np.min(timings)),
        'maximum_s': float(np.max(timings)),
        'mean_s': float(np.mean(timings)),
        'std_s': float(np.std(timings)),
        'q25_s': float(q25_s),
        'q75_s': float(q75_s),
        'iqr_s': float(q75_s - q25_s),
        'times_s': [float(x) for x in timings],
    }


def benchmark_call(fn, *, warmups=1, repeats=3):
    # Warm, synchronize, and return all individual runtimes.
    warmups = max(0, int(warmups))
    repeats = max(1, int(repeats))

    result = None
    for _ in range(warmups):
        result = fn()
        block_result(result)

    timings = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        result = fn()
        block_result(result)
        timings.append(time.perf_counter() - t0)

    return result, summarize_timings(timings)


def compare_device_batches(candidate, reference):
    # Compare two device-resident six-link batches using scalar reductions.
    candidate = jnp.asarray(candidate)
    reference = jnp.asarray(reference)

    delta = candidate - reference
    max_abs_value = jnp.max(jnp.abs(delta))
    ref_norm = jnp.linalg.norm(reference)
    rel_l2_value = jnp.where(
        ref_norm > 0,
        jnp.linalg.norm(delta) / ref_norm,
        jnp.nan,
    )

    max_abs_value.block_until_ready()
    rel_l2_value.block_until_ready()

    return {
        'max_abs': float(max_abs_value),
        'rel_l2': float(rel_l2_value),
    }


def assess_paired_candidate(
    baseline_times,
    candidate_times,
    *,
    rel_l2,
    min_improvement_percent=MIN_MEDIAN_IMPROVEMENT_PERCENT,
    max_rel_l2=MAX_REL_L2,
):
    baseline = summarize_timings(baseline_times)
    candidate = summarize_timings(candidate_times)

    runtime_change_percent = (
        100.0 * (candidate['median_s'] / baseline['median_s'] - 1.0)
    )
    improvement_percent = -runtime_change_percent
    numerically_valid = bool(np.isfinite(rel_l2) and rel_l2 <= max_rel_l2)

    # Conservative distributional guard:
    # at least 75% of candidate timings must be below at least 75% of baseline
    # timings. Equivalently, the central 50% intervals do not overlap.
    iqr_separated = bool(candidate['q75_s'] < baseline['q25_s'])

    if (
        numerically_valid
        and improvement_percent >= min_improvement_percent
        and iqr_separated
    ):
        decision = 'adopt'
    elif numerically_valid and improvement_percent >= min_improvement_percent:
        decision = 'retest'
    else:
        decision = 'retain_baseline'

    return {
        'baseline': baseline,
        'candidate': candidate,
        'runtime_change_percent': float(runtime_change_percent),
        'improvement_percent': float(improvement_percent),
        'numerically_valid': numerically_valid,
        'iqr_separated': iqr_separated,
        'decision': decision,
    }


def make_shift_kwargs(
    row_chunk_size,
    lag_block_size,
    *,
    backend='lagfirst_chunked_lagblock',
    job_block_size=1,
):
    kwargs = dict(BASE_SHIFT_KWARGS)
    kwargs.update({
        'assembly_backend': str(backend),
        'row_chunk_size': int(row_chunk_size),
        'lag_block_size': int(lag_block_size),
        'job_block_size': int(job_block_size),
    })
    return kwargs


def build_one_arm_plan(grid_context, kwargs):
    t0 = time.perf_counter()
    plan = OneArmWDMPlan.build_from_context(
        grid_context,
        orbits_trailing,
        theta,
        phi,
        L_trunc=L_TRUNC,
        variable_shift_kwargs=kwargs,
        use_jax=True,
    )
    build_s = time.perf_counter() - t0
    return plan, float(build_s)


def show_rows(rows, *, sort_by=None):
    if pd is not None:
        frame = pd.DataFrame(rows)
        if sort_by is not None and sort_by in frame.columns:
            frame = frame.sort_values(sort_by, kind='stable').reset_index(drop=True)
        display(frame)
        return frame

    rows_out = list(rows)
    if sort_by is not None:
        rows_out = sorted(rows_out, key=lambda row: row.get(sort_by, np.inf))
    for row in rows_out:
        print(row)
    return rows_out


def save_rows(rows, filename):
    if not SAVE_RESULTS:
        return
    if pd is None:
        print('pandas is unavailable; skipping CSV export:', filename)
        return
    pd.DataFrame(rows).to_csv(filename, index=False)
    print('Saved:', filename)


## 4. Generate the fixed waveform and orbit object

This is identical in spirit to the production getting-started case. The same
waveform is reused for every candidate. Its generation time is not part of the
shift benchmark.


In [ ]:
# --- 4a. Orbit and waveform generator ---

td_gen = GenerateEMRIWaveform(
    'FastKerrEccentricEquatorialFlux',
    sum_kwargs=dict(pad_output=True, odd_len=True),
    return_list=True,
)

orbit_data = read_orbit_file(ORBIT_FILE)
orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
orbits_trailing = InterpolatedOrbits(
    orbit_times,
    orbit_data.positions,
    orbit_data.velocities,
)

t0 = orbit_data.t0 + 1.0 * YRSID_SI

model_kwargs = {
    'td_gen': td_gen,
    'm1': m1,
    'm2': m2,
    'a': a,
    'p0': p0,
    'e0': e0,
    'x0': x0,
    'dist': dist,
    'theta': theta,
    'phi': phi,
    'qK': qK,
    'phiK': phiK,
    'Phi_phi0': Phi_phi0,
    'Phi_theta0': Phi_theta0,
    'Phi_r0': Phi_r0,
    'eps': eps,
}

window_kwargs = {
    'alpha': alpha,
    'tukey_factor': tukey_factor,
}

print('Orbit samples:', orbit_data.size)
print('Starting waveform generation...')


In [ ]:
# --- 4b. Generate and pad the waveform ---

t_wave0 = time.perf_counter()

times_pad, hplus_td, hcross_td, pad_info = generate_windowed_padded_waveform(
    waveform_model='emri',
    t0=t0,
    dt=dt,
    N=N,
    model_kwargs=model_kwargs,
    window_function='tukey',
    window_kwargs=window_kwargs,
    wdm_Nf=Nf,
    return_padding_info=True,
)

waveform_generation_s = time.perf_counter() - t_wave0

print('Waveform generation_s:', f'{waveform_generation_s:.3f}')
print('times_pad:', times_pad.shape, times_pad.dtype)
print('hplus_td:', hplus_td.shape, hplus_td.dtype)
print('hcross_td:', hcross_td.shape, hcross_td.dtype)
print('WDM Nt x Nf:', pad_info['wdm_Nt'], 'x', Nf)
print('Tail right padding:', pad_info['wdm_tail_pad_right'])
print('RSS:', f'{rss_gib():.2f} GiB')


## 5. Build the WDM grid and perform the two DWTs once

The WDM grid and waveform coefficients are shared by all candidates. The
reported candidate timings therefore isolate the prepared six-link
materialized one-arm calculation.


In [ ]:
# --- 5. Shared WDM grid and waveform context ---

t_grid0 = time.perf_counter()
grid_context = _build_wdm_dT_grid(
    times_pad,
    Nf=Nf,
    d=d_wdm,
    A_frac=A_frac,
    q=q_wdm,
    calc_m0=True,
)
grid_build_s = time.perf_counter() - t_grid0

t_dwt0 = time.perf_counter()
wave_context = _build_wdm_dT_context_from_wdm(
    times_pad,
    hplus_td,
    hcross_td,
    grid_context['wdm'],
)
block_result((wave_context['w_t_plus'], wave_context['w_t_cross']))
two_dwt_s = time.perf_counter() - t_dwt0

wdm = grid_context['wdm']
w_plus_device = jnp.asarray(wave_context['w_t_plus'], dtype=jnp.complex64)
w_cross_device = jnp.asarray(wave_context['w_t_cross'], dtype=jnp.complex64)
block_result((w_plus_device, w_cross_device))

print('WDM grid build_s:', f'{grid_build_s:.3f}')
print('Two DWTs + synchronization_s:', f'{two_dwt_s:.3f}')
print('WDM Nt:', wdm.Nt)
print('WDM Nf:', wdm.Nf)
print('WDM N:', wdm.N)
print('WDM dT:', wdm.dT)
print('WDM dF:', wdm.dF)
print('plus WDM:', w_plus_device.shape, w_plus_device.dtype)
print('cross WDM:', w_cross_device.shape, w_cross_device.dtype)
print('RSS:', f'{rss_gib():.2f} GiB')


## 6. Build the production plan and materialize the fixed six-link input

The plus/cross polarization combination is formed once on the JAX device and is excluded from both timed routes. This isolates the variable-shift kernel. The same materialized six-link array is supplied to the production and contiguous-$T_l/T_p$ implementations.


In [ ]:
# --- 6. Production plan, materialized input, and baseline reference ---

baseline_kwargs = make_shift_kwargs(
    ROW_CHUNK_SIZE,
    LAG_BLOCK_SIZE,
)

baseline_plan, baseline_plan_build_s = build_one_arm_plan(
    grid_context,
    baseline_kwargs,
)
shift_plan = baseline_plan.shift_plan

assert shift_plan.resolved_assembly_backend == 'lagfirst_chunked_lagblock'
assert shift_plan.resolved_assembly_precision == 'complex64'
assert shift_plan.Nt == ROW_CHUNK_SIZE

# Reproduce the exact production materialization path once, outside timing.
source_coefficients = baseline_plan._waveform_sources_device(
    w_plus_device,
    w_cross_device,
)
amplitude_weights = baseline_plan._device_amplitude_weights()
unshifted_device = _combine_polarizations_device(
    amplitude_weights[:, 0],
    amplitude_weights[:, 1],
    source_coefficients[0],
    source_coefficients[1],
)
block_result(unshifted_device)

# Populate the ordinary persistent device-plan cache before timing.
reference_device = shift_plan.apply_device(unshifted_device)
block_result(reference_device)


def run_baseline():
    return shift_plan.apply_device(unshifted_device)


_, baseline_timing = benchmark_call(
    run_baseline,
    warmups=CONFIRM_WARMUPS,
    repeats=CONFIRM_REPEATS,
)

# Confirm that the isolated shift output is the same output returned by the
# complete materialized one-arm route.
complete_materialized = baseline_plan.apply_batch_device_materialized(
    w_plus_device,
    w_cross_device,
)
block_result(complete_materialized)
complete_comparison = compare_device_batches(
    complete_materialized,
    reference_device,
)

print('Baseline kwargs:', baseline_kwargs)
print('Baseline plan build_s:', f'{baseline_plan_build_s:.3f}')
print('Materialized input:', unshifted_device.shape, unshifted_device.dtype)
print('Baseline shift median_s:', f"{baseline_timing['median_s']:.3f}")
print('Baseline shift q25_s:', f"{baseline_timing['q25_s']:.3f}")
print('Baseline shift q75_s:', f"{baseline_timing['q75_s']:.3f}")
print('Baseline shift IQR_s:', f"{baseline_timing['iqr_s']:.3f}")
print('Baseline all:', [round(x, 3) for x in baseline_timing['times_s']])
print('Reference shape:', reference_device.shape)
print('Reference dtype:', reference_device.dtype)
print('Complete-route rel_l2:', f"{complete_comparison['rel_l2']:.3e}")
print(
    'Ordinary persistent plan memory_MiB:',
    shift_plan.device_plan_memory_bytes / 1024**2,
)
print('RSS:', f'{rss_gib():.2f} GiB')


## 7. Prepare and validate the contiguous $T_l/T_p$ layout

The reversal and device transfer are one-time plan preparation operations and are completed before compilation, warm-up, or timing. For the current symmetric lag range, `-ell + offset` must equal the reversed column sequence.


In [ ]:
# --- 7. Prepare the experimental Tl/Tp consumption-order cache ---

ell_host = np.asarray(shift_plan.ell_all, dtype=np.int64)
j_neg_host = -ell_host + int(shift_plan.offset)
expected_reverse = np.arange(
    ell_host.size - 1,
    -1,
    -1,
    dtype=np.int64,
)

assert np.array_equal(j_neg_host, expected_reverse), (
    'Unexpected lag ordering: the contiguous experiment assumes that '
    '-ell + offset is an exact reversal of the stored Tl/Tp columns.'
)

prepare_t0 = time.perf_counter()
tltp_cache = shift_plan.prepare_device_tltp_consumption(
    synchronise=True,
)
prepare_tltp_s = time.perf_counter() - prepare_t0

print('ell range:', int(ell_host[0]), 'to', int(ell_host[-1]))
print('number of lags:', ell_host.size)
print('padded number of lags:', tltp_cache['padded_n_lag'])
print('one-time contiguous cache build_s:', f'{prepare_tltp_s:.3f}')
print(
    'additional contiguous cache memory_MiB:',
    shift_plan.tltp_consumption_cache_memory_bytes / 1024**2,
)
print(
    'plan with both layouts memory_MiB:',
    shift_plan.device_plan_with_tltp_consumption_memory_bytes / 1024**2,
)
print('Tl consumption:', tltp_cache['Tl_consumption'].shape)
print('Tp consumption:', tltp_cache['Tp_consumption'].shape)


def run_candidate():
    return shift_plan.apply_device_contiguous_tltp(unshifted_device)


# Compile the candidate and validate it before the paired benchmark.
for _ in range(CONFIRM_WARMUPS):
    block_result(run_candidate())

candidate_reference = run_candidate()
block_result(candidate_reference)
comparison = compare_device_batches(
    candidate_reference,
    reference_device,
)

print('Candidate max_abs:', f"{comparison['max_abs']:.6e}")
print('Candidate rel_l2:', f"{comparison['rel_l2']:.6e}")
assert comparison['rel_l2'] <= MAX_REL_L2


## 8. Alternating paired confirmation

Both kernels are already compiled and warmed. Fifteen paired rounds alternate execution order to reduce drift and ordering bias. The candidate is adopted only if it is numerically valid, at least 3% faster in median, and its third quartile is below the baseline first quartile.


In [ ]:
# --- 8. Production gather versus contiguous Tl/Tp slices ---

# Rewarm both routes immediately before the paired measurement.
for _ in range(CONFIRM_WARMUPS):
    block_result(run_baseline())
    block_result(run_candidate())

baseline_pair_times = []
candidate_pair_times = []

for repeat_index in range(CONFIRM_REPEATS):
    if repeat_index % 2 == 0:
        order = (
            ('baseline', run_baseline),
            ('candidate', run_candidate),
        )
    else:
        order = (
            ('candidate', run_candidate),
            ('baseline', run_baseline),
        )

    for label, fn in order:
        t0 = time.perf_counter()
        output = fn()
        block_result(output)
        elapsed = time.perf_counter() - t0

        if label == 'baseline':
            baseline_pair_times.append(float(elapsed))
        else:
            candidate_pair_times.append(float(elapsed))

assessment = assess_paired_candidate(
    baseline_pair_times,
    candidate_pair_times,
    rel_l2=comparison['rel_l2'],
)
baseline_stats = assessment['baseline']
candidate_stats = assessment['candidate']

result_row = {
    'experiment': 'contiguous_tltp',
    'row_chunk_size': ROW_CHUNK_SIZE,
    'lag_block_size': LAG_BLOCK_SIZE,
    'baseline_median_s': baseline_stats['median_s'],
    'baseline_q25_s': baseline_stats['q25_s'],
    'baseline_q75_s': baseline_stats['q75_s'],
    'baseline_iqr_s': baseline_stats['iqr_s'],
    'candidate_median_s': candidate_stats['median_s'],
    'candidate_q25_s': candidate_stats['q25_s'],
    'candidate_q75_s': candidate_stats['q75_s'],
    'candidate_iqr_s': candidate_stats['iqr_s'],
    'speedup': baseline_stats['median_s'] / candidate_stats['median_s'],
    'runtime_change_percent': assessment['runtime_change_percent'],
    'improvement_percent': assessment['improvement_percent'],
    'iqr_separated': assessment['iqr_separated'],
    'numerically_valid': assessment['numerically_valid'],
    'decision': assessment['decision'],
    'max_abs': comparison['max_abs'],
    'rel_l2': comparison['rel_l2'],
    'one_time_cache_build_s': prepare_tltp_s,
    'additional_cache_mib': (
        shift_plan.tltp_consumption_cache_memory_bytes / 1024**2
    ),
    'baseline_times_s': baseline_stats['times_s'],
    'candidate_times_s': candidate_stats['times_s'],
}

show_rows([result_row])
save_rows([result_row], f'{RESULTS_PREFIX}_confirm.csv')

print('\nProduction gather:')
print(
    f"  median={baseline_stats['median_s']:.3f} s | "
    f"IQR={baseline_stats['iqr_s']:.3f} s | "
    f"[q25,q75]=[{baseline_stats['q25_s']:.3f}, "
    f"{baseline_stats['q75_s']:.3f}]"
)
print('  all:', [round(x, 3) for x in baseline_stats['times_s']])

print('\nContiguous Tl/Tp:')
print(
    f"  median={candidate_stats['median_s']:.3f} s | "
    f"IQR={candidate_stats['iqr_s']:.3f} s | "
    f"[q25,q75]=[{candidate_stats['q25_s']:.3f}, "
    f"{candidate_stats['q75_s']:.3f}]"
)
print('  all:', [round(x, 3) for x in candidate_stats['times_s']])

print('\nRuntime change:', f"{assessment['runtime_change_percent']:+.2f}%")
print('Speedup:', f"{result_row['speedup']:.3f}x")
print('IQR separated:', assessment['iqr_separated'])
print('Numerically valid:', assessment['numerically_valid'])
print('max_abs:', f"{comparison['max_abs']:.6e}")
print('rel_l2:', f"{comparison['rel_l2']:.6e}")
print('Decision:', assessment['decision'])

if assessment['decision'] == 'adopt':
    print(
        '\nThe candidate passes the experimental adoption guard. '
        'Do not switch production automatically; first integrate it behind '
        'the normal public wrapper and rerun the end-to-end profiler.'
    )
elif assessment['decision'] == 'retest':
    print(
        '\nThe median gain is large enough, but the timing distributions '
        'overlap. Repeat on an otherwise idle desktop before deciding.'
    )
else:
    print(
        '\nRetain the current production gather implementation. The '
        'contiguous layout does not justify its additional persistent memory.'
    )


## 9. Memory and promotion discipline

During this experiment both $T_l/T_p$ layouts remain cached so they can be compared in the same process. This deliberately increases persistent memory. A promoted implementation would retain only the winning layout.


In [ ]:
# --- 9. Final experiment summary ---

print('Production schedule:', (ROW_CHUNK_SIZE, LAG_BLOCK_SIZE))
print(
    'Ordinary device-plan memory_MiB:',
    shift_plan.device_plan_memory_bytes / 1024**2,
)
print(
    'Experimental duplicate Tl/Tp memory_MiB:',
    shift_plan.tltp_consumption_cache_memory_bytes / 1024**2,
)
print(
    'Total while comparing both layouts_MiB:',
    shift_plan.device_plan_with_tltp_consumption_memory_bytes / 1024**2,
)
print('Decision:', result_row['decision'])


## 10. Benchmark discipline

1. Restart the kernel after replacing the source files.
2. Run on an otherwise idle desktop.
3. Run the notebook from the repository environment that imports the edited WDM package.
4. Keep the fixed `32640/17` schedule and full-year input.
5. Report the complete baseline and candidate timing lists, quartiles, relative $L_2$ difference, memory overhead, and final decision.
6. Do not alter the normal production wrapper unless the candidate passes the dedicated benchmark.
